# 8 月逐 Part 促销审核模型

## TL;DR

以 2026-07-28 生产订单、认证成本、库存快照和 Catalog 状态为依据，21 个 Part 中建议提报 15 个、暂缓 6 个。促销兼容广告预算建议由原销售计划的 $4,050 收紧至 $2,700；60% 促销订单占比主情景的广告后利润率约 12.45%，70% 压力情景约 11.12%。紫鸟提交保持锁定，等待人工审核。

## Context & Methods

- 目标：8 月 150 单，店铺广告后利润率 10%–15%，经营目标约 12%。
- 价格：使用 6–7 月最新/保守 WSC 成交价；没有价格证据的 Part 暂缓。
- 成本：只使用生产成本表中已认证 USD 单位成本；缺成本暂缓。
- 折扣：B2B 总折扣是最终折扣；数量 Offer 与 B2B 折扣按乘法叠加。
- 角色底线：跑量/受控增长 12%，利润池/修复自然款 20%；全店广告后硬底线仍为 10%。
- 促销组合模型：假设合格订单中 60% 或 70% 命中所建模的最差折扣；100% 用于极端压力测试。
- 未知：真实活动接受价、Wayfair 是否叠加额外平台资助、提报后实际促销订单占比。上述变量必须在后台提交前复核。

In [1]:
from pathlib import Path
import csv

source_path = Path('analysis/august-promotion-source-2026-07-28.csv')
with source_path.open(encoding='utf-8', newline='') as handle:
    rows = list(csv.DictReader(handle))

numeric_fields = ['price_basis_cents', 'cost_cents', 'inventory_on_hand', 'catalog_live_count', 'june_units', 'july_units', 'b2c_discount', 'b2b_total_discount', 'quantity_offer', 'worst_discount', 'margin_floor']
for row in rows:
    for field in numeric_fields:
        row[field] = None if row[field] == '' else float(row[field])

len(rows)

21

## Data quality

In [2]:
quality = {
    'parts': len(rows),
    'certified_cost_covered': sum(row['cost_cents'] is not None for row in rows),
    'price_basis_covered': sum(row['price_basis_cents'] is not None for row in rows),
    'inventory_covered': sum(row['inventory_on_hand'] is not None for row in rows),
    'catalog_live_covered': sum((row['catalog_live_count'] or 0) >= 1 for row in rows),
    'duplicate_catalog_mappings': sum((row['catalog_live_count'] or 0) > 1 for row in rows),
}
quality

{'parts': 21,
 'certified_cost_covered': 19,
 'price_basis_covered': 17,
 'inventory_covered': 21,
 'catalog_live_covered': 21,
 'duplicate_catalog_mappings': 1}

## Results

In [3]:
for row in rows:
    if row['action'] == 'PROPOSE':
        net_price = row['price_basis_cents'] * (1 - row['worst_discount'])
        row['worst_margin'] = (net_price - row['cost_cents']) / net_price
        assert row['worst_margin'] + 1e-9 >= row['margin_floor'], row['part']
    else:
        row['worst_margin'] = None

decision_counts = {decision: sum(row['action'] == decision for row in rows) for decision in ['PROPOSE', 'HOLD']}
decision_counts

{'PROPOSE': 15, 'HOLD': 6}

In [4]:
baseline_revenue = 17648.94
baseline_gross_profit = 6167.66
full_promo_discount_loss = 2419.56
recommended_ads = 2700.00
fallback_ads = 2200.00

def scenario(promo_share, ads=recommended_ads):
    discount_loss = full_promo_discount_loss * promo_share
    revenue = baseline_revenue - discount_loss
    gross_profit = baseline_gross_profit - discount_loss
    post_ad_profit = gross_profit - ads
    return {
        'promo_share': promo_share,
        'revenue': round(revenue, 2),
        'gross_profit': round(gross_profit, 2),
        'ads': round(ads, 2),
        'post_ad_profit': round(post_ad_profit, 2),
        'post_ad_margin': round(post_ad_profit / revenue, 4),
        'ad_cap_at_10pct': round(gross_profit - revenue * 0.10, 2),
        'ad_cap_at_12pct': round(gross_profit - revenue * 0.12, 2),
    }

scenarios = [scenario(share) for share in (0.60, 0.70, 1.00)]
assert 0.10 <= scenarios[0]['post_ad_margin'] <= 0.15
assert scenarios[1]['post_ad_margin'] >= 0.10
scenarios

[{'promo_share': 0.6,
  'revenue': 16197.2,
  'gross_profit': 4715.92,
  'ads': 2700.0,
  'post_ad_profit': 2015.92,
  'post_ad_margin': 0.1245,
  'ad_cap_at_10pct': 3096.2,
  'ad_cap_at_12pct': 2772.26},
 {'promo_share': 0.7,
  'revenue': 15955.25,
  'gross_profit': 4473.97,
  'ads': 2700.0,
  'post_ad_profit': 1773.97,
  'post_ad_margin': 0.1112,
  'ad_cap_at_10pct': 2878.44,
  'ad_cap_at_12pct': 2559.34},
 {'promo_share': 1.0,
  'revenue': 15229.38,
  'gross_profit': 3748.1,
  'ads': 2700.0,
  'post_ad_profit': 1048.1,
  'post_ad_margin': 0.0688,
  'ad_cap_at_10pct': 2225.16,
  'ad_cap_at_12pct': 1920.57}]

## Takeaways

1. 15 个 Part 可进入审核，6 个因为缺价、缺成本、低库存或重复 Catalog 映射而暂缓。
2. 原 $4,050 广告池没有为促销折扣留空间；审核版采用 $2,700（基础 $2,200 + 赢家机动 $500）。
3. 若促销订单占比超过 70%、平台进一步压价或组合毛利低于模型，广告预算立即收缩到 $2,200；全量促销极端情景的 10% 硬上限约为 $2,225。
4. MFC-D3-B 是最薄的受控跑量款；VFC-2B/W 只接受 8% B2B，不接受 13%。
5. 所有行保持 `PENDING_REVIEW` 且 `canSubmitToZiniao=false`，人工审核后才允许进入紫鸟。